# Fire research: SwinUNETR on Kaggle

Converted from Colab export. In the Kaggle notebook settings, enable **Internet** and an
**accelerator** that matches `ACCELERATOR`: a GPU for `'gpu'` (default), a TPU VM for `'tpu'`. Under **Add-ons → Secrets**, add `WANDB` (your W&B API
key) and `HF_TOKEN` (your Hugging Face read token), and grant this notebook access.
`HF_TOKEN` is optional for a public dataset or when using attached Kaggle data.

Run the cells in order. Loss and learning rate follow the TS-SatFire paper's prediction setup
(`LOSS = 'dicece'`: 1:1 Dice + cross entropy, class weights 1.0 / 446.7836; `LEARNING_RATE = 0.001`).
Other defaults follow the paper's SwinUNETR-3D prediction setup: batch size 8,
`TS_LENGTH = 6` input days, feature size 36, and 100 epochs (3 attention heads; the paper gives no head count).
**New, not in the paper:** `USE_DICE_FOCAL = True` trains with Dice + focal loss (`--loss dicefocal`,
weights `LAMBDA_DICE` / `LAMBDA_FOCAL`) instead of `LOSS`. `LR_SCHEDULE` optionally decays the learning rate (`'plateau'` or `'cosine'`);
the default `'none'` keeps the paper's constant rate. Each run logs a `Learning Rate` chart.
**New, not in the paper:** `ACCELERATOR = 'tpu'` trains on one TPU core (v3-8) or chip (v5e-8) through
PyTorch/XLA (`--device tpu`); the other cores stay idle, the first steps are slow while XLA compiles,
and results are close to, not identical to, a GPU run.
Set `EPOCHS = 1` for an initial end-to-end check. No GPU memory capacity is assumed.

The training code is cloned from GitHub (`REPO_BRANCH`, optionally pinned to `REPO_COMMIT`) and run as
`python -m firepred.cli.train`; this notebook only holds settings. `MODEL_NAME` picks any model registered in
`firepred/models/zoo/` (`MODEL_OPTIONS` passes its extra settings as `--model-option KEY=VALUE`).
With `LOSS = 'dicece'`, `CE_WEIGHT`, `LAMBDA_DICE` and `LAMBDA_CE` set the Dice + cross-entropy loss
(paper defaults: 1.0 / 446.7836 and 1:1); they are passed to the trainer and recorded in W&B.
`USE_TESSERA` switches the experiment:
- `False`: 27 stored channels, 43 model channels; downloads only `data.zip`.
- `True`: also downloads `tessera.zip` from `SamuelWu318/ts-tesserafire`, drops the fires
  without prior-year TESSERA (all 2017 fires), and trains with 172 model channels.
  To train the matched no-TESSERA baseline on the same fires, keep `USE_TESSERA = True`
  and set `TESSERA_INPUTS = False`.

Data and code go under `/kaggle/temp/fire-research`; W&B logs and model checkpoints
go under `/kaggle/working/fire-research-output`. Save a notebook version to retain
its working-directory outputs. Raw data is not included in saved outputs.
Each epoch logs to W&B, plotted against the epoch: the training/validation loss (with its
weighted Dice and cross-entropy parts when `LOSS = 'dicece'`), fire-growth and background F1,
validation mean IoU and mean Dice. The run config records the loss, its class weights and the
learning rate. Each best checkpoint is also uploaded as a versioned W&B **model Artifact** under
`gt-fri/AFBAPred`. Training waits for that upload to finish. The `best` alias points
to the best checkpoint so far; earlier uploaded versions remain available.


In [11]:
from pathlib import Path
import os
import sys
import subprocess

# Experiment settings: every one maps to a flag of `python -m firepred.cli.train`.
ACCELERATOR = 'gpu'     # 'gpu' or 'tpu'; must match Settings -> Accelerator.
                        # NEW: 'tpu' trains on one TPU core/chip via PyTorch/XLA (--device tpu).
DEVICE = 'tpu' if ACCELERATOR == 'tpu' else 'cuda'  # Trainer --device.
MODEL_NAME = 'swinunetr3d'  # Any registered model: swinunetr3d, swinunetr3d_tpool, unet3d, attunet, ...
MODEL_OPTIONS = {}         # Model-specific settings, e.g. {'width': 32}; passed as --model-option KEY=VALUE.
BATCH_SIZE = 8          # Paper's SwinUNETR-3D prediction batch size; reduce if CUDA runs out of memory.
EPOCHS = 100            # Set to 1 for an initial training/validation check.
LEARNING_RATE = 0.001   # Adam learning rate; the paper's prediction setting is 0.001.
USE_DICE_FOCAL = False  # NEW, not in the paper: True trains with Dice + focal ('dicefocal') instead of LOSS.
LOSS = 'dicece'         # Used when USE_DICE_FOCAL is False. 'dicece': paper's 1:1 Dice + CE;
                        # 'dice': released-code Dice loss.
# Dice + CE settings (used only with LOSS = 'dicece'; LAMBDA_DICE also with USE_DICE_FOCAL); defaults are the paper's.
CE_WEIGHT = (1.0, 446.7836)  # Class weights (background, fire); MONAI applies them to Dice and CE.
CE_WEIGHT = (1.0, 344.7)
CE_WEIGHT = (1.0, 457.1)     # NOTE: assigned three times; this last value is the one used.
LAMBDA_DICE = 1.0       # Weight of the Dice term.
LAMBDA_CE = 1.0         # Weight of the cross-entropy term (1:1 with Dice in the paper).
# Focal settings (used only with USE_DICE_FOCAL = True). Unweighted Dice starts near 0.7 and focal near 0.1.
LAMBDA_FOCAL = 1.0      # Weight of the focal term.
FOCAL_GAMMA = 2.0       # Focusing exponent; 0 = plain binary cross entropy.
FOCAL_ALPHA = None      # Weight of fire pixels in the focal term, e.g. 0.25; None = off.
if USE_DICE_FOCAL:
    LOSS = 'dicefocal'  # LAMBDA_DICE x Dice + LAMBDA_FOCAL x focal loss on the fire channel, no CE.
# NEW, NOT IN THE TS-SATFIRE PAPER OR THE OFFICIAL CODE: optional learning-rate decay.
# 'none' keeps the paper's constant LEARNING_RATE. 'plateau' multiplies it by LR_DECAY_FACTOR
# after LR_PATIENCE epochs without a lower validation loss; 'cosine' decays it smoothly to
# LR_MIN over EPOCHS. Runs with decay get '_lr-<schedule>' in their W&B name and checkpoints.
LR_SCHEDULE = 'none'    # NEW: 'none' (paper) | 'plateau' | 'cosine'
LR_DECAY_FACTOR = 0.5   # NEW, plateau only: multiply the learning rate by this.
LR_PATIENCE = 5         # NEW, plateau only: epochs without improvement before decaying.
LR_MIN = 1e-6           # NEW, plateau and cosine: the lowest learning rate.
# NEW, NOT IN THE TS-SATFIRE PAPER: score the test fires with the current weights every
# TEST_EVERY epochs and chart test F1/IoU. Logging only: no gradients, eval mode, and the
# best checkpoint is still chosen by validation loss. 0 turns it off.
TEST_EVERY = 10         # NEW: epochs between test checks during training.
NUM_HEADS = 3           # Attention heads in each Swin stage.
FEATURE_SIZE = 36       # Paper's feature size (24 and 36 tested, 36 picked); must be divisible by NUM_HEADS.
NUM_WORKERS = 2         # Each worker builds whole batches in host RAM; 4 ran Kaggle out of RAM with TESSERA.
UNZIP_DIR = 'off'       # 'off': read the window archives; 'auto': unzip once beside the data (~5.5x disk).
RUN_GPU_SMOKE_TEST = True  # Test one training step before the full experiment.

# Window settings; must match how the window data was built (python -m firepred.cli.dataset process -ts/-it).
# The TS-SatFire paper's prediction benchmark uses 6 input days.
TS_LENGTH = 6           # Input days per window.
TS_INTERVAL = 1         # Days between window starts.

# TESSERA switch. Only USE_TESSERA = True downloads or reads any TESSERA data.
USE_TESSERA = True     # True: drop 2017 fires and load per-fire TESSERA embeddings.
TESSERA_INPUTS = True   # With USE_TESSERA: False trains the matched baseline (same fires, 43 channels).
MODEL_CHANNELS = 172 if (USE_TESSERA and TESSERA_INPUTS) else 43  # 43 base + 128 TESSERA + 1 mask.

# Training code is cloned from GitHub; set REPO_COMMIT to pin an exact version.
REPO_URL = 'https://github.com/SamuelWu318/fire-research-and-innovation.git'
REPO_BRANCH = 'explain'  # Branch with the firepred package; 'main' once it is merged.
REPO_COMMIT = None      # Example: 'd252458'

WANDB_ENTITY = 'gt-fri' # Team with write permission for your account.
WANDB_PROJECT = 'AFBAPred'  # Already configured in the bundled trainer.
WANDB_SECRET = 'WANDB'  # Change to WANDB_API_KEY if that is your Kaggle secret label.
HF_SECRET = 'HF_TOKEN'  # Read token for private/gated Hugging Face data.
# None: the repo for TS_LENGTH / TS_INTERVAL, from firepred.config.data.window_repo_for after the clone
# (SamuelWu318/ts-satfire-processed-{TS_LENGTH}-{TS_INTERVAL}). Set a repo id to override.
HF_DATASET = None                                       # data.zip
TESSERA_DATASET = 'SamuelWu318/ts-tesserafire'           # tessera.zip (TESSERA mode only)

# Leave both unset to download data.zip from the Hugging Face dataset above.
# Alternatively, attach a Kaggle Dataset and set ONE of these absolute paths.
ATTACHED_DATA_ROOT = None  # Example: '/kaggle/input/my-fire-data/data'
ATTACHED_ZIP = None        # Example: '/kaggle/input/my-fire-data/data.zip'
# TESSERA mode only: leave unset to download tessera.zip, or set ONE of these.
ATTACHED_TESSERA_DIR = None  # Folder with p<id>.npz and excluded_npz.txt
ATTACHED_TESSERA_ZIP = None  # Example: '/kaggle/input/my-tessera/tessera.zip'

TEMP_ROOT = Path('/kaggle/temp/fire-research')
CODE_DIR = TEMP_ROOT / 'code'
EXTRACT_DIR = TEMP_ROOT / f'data-ts{TS_LENGTH}-it{TS_INTERVAL}'
TESSERA_EXTRACT_DIR = TEMP_ROOT / 'tessera'
TESSERA_VIEW_DIR = TEMP_ROOT / f'data-ts{TS_LENGTH}-it{TS_INTERVAL}-tessera-view'  # Symlinks: kept fires + tessera/
DOWNLOAD_DIR = TEMP_ROOT / 'downloads'
OUTPUT_DIR = Path('/kaggle/working/fire-research-output')
for folder in (EXTRACT_DIR, DOWNLOAD_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

assert ACCELERATOR in ('gpu', 'tpu'), "ACCELERATOR must be 'gpu' or 'tpu'"
assert BATCH_SIZE >= 1 and EPOCHS >= 1 and NUM_WORKERS >= 0
assert LOSS in ('dice', 'dicece', 'dicefocal')
assert LR_SCHEDULE in ('none', 'plateau', 'cosine') and 0 < LR_DECAY_FACTOR < 1 and LR_PATIENCE >= 0
assert TEST_EVERY >= 0
assert len(CE_WEIGHT) == 2 and min(CE_WEIGHT) >= 0 and LAMBDA_DICE >= 0 and LAMBDA_CE >= 0
assert NUM_HEADS > 0 and FEATURE_SIZE > 0 and FEATURE_SIZE % NUM_HEADS == 0
assert not (ATTACHED_DATA_ROOT and ATTACHED_ZIP), 'Choose a data folder OR a ZIP.'
assert not (ATTACHED_TESSERA_DIR and ATTACHED_TESSERA_ZIP), 'Choose a TESSERA folder OR a ZIP.'
print('accelerator:', ACCELERATOR, '| model:', MODEL_NAME, MODEL_OPTIONS or '', '| USE_DICE_FOCAL:', USE_DICE_FOCAL, '| USE_TESSERA:', USE_TESSERA, '| TESSERA inputs:', USE_TESSERA and TESSERA_INPUTS,
      '| model channels:', MODEL_CHANNELS, '| loss:', LOSS,
      *(('| CE weight:', CE_WEIGHT, '| lambda dice/ce:', LAMBDA_DICE, LAMBDA_CE) if LOSS == 'dicece' else ()),
      *(('| lambda dice/focal:', LAMBDA_DICE, LAMBDA_FOCAL, '| focal gamma/alpha:', FOCAL_GAMMA, FOCAL_ALPHA)
        if LOSS == 'dicefocal' else ()),
      '| lr:', LEARNING_RATE, '| lr schedule (NEW, not in paper):', LR_SCHEDULE, '| test every (NEW):', TEST_EVERY, '| code:', REPO_BRANCH,
      '| windows:', TS_LENGTH, 'days every', TS_INTERVAL,
      '| data:', HF_DATASET or 'repo for these windows')


USE_TESSERA: False | TESSERA inputs: False | model channels: 43 | loss: dicece | CE weight: (1.0, 446.7836) | lambda dice/ce: 1.0 1.0 | lr: 0.001 | code: main | windows: 10 days every 3 | data: SamuelWu318/ts-satfire-processed-window


## 1. Install the training dependencies
Keep Kaggle's installed PyTorch/CUDA/torchvision combination. The Colab apt repairs,
GDAL build, geospatial processing packages, and Airflow installation are not required
to train from already-processed NPZ files. On a TPU VM (`ACCELERATOR = 'tpu'`) the installed
PyTorch and PyTorch/XLA (`torch_xla`) are kept the same way. MONAI 1.3.2 matches the supplied setup. TESSERA mode needs no extra packages.


In [3]:
import importlib.metadata
import importlib.util

import torch

try:
    import torchvision
    TORCHVISION_VERSION = torchvision.__version__
except ImportError:  # Not every TPU image ships torchvision; the trainer does not need it.
    TORCHVISION_VERSION = None

pins = [f'torch=={torch.__version__}']
if TORCHVISION_VERSION:
    pins.append(f'torchvision=={TORCHVISION_VERSION}')
if ACCELERATOR == 'tpu':
    # Check PyTorch/XLA without opening the TPU here: only one process can hold it,
    # and the training subprocess needs it.
    if importlib.util.find_spec('torch_xla') is None:
        raise RuntimeError('No torch_xla: set Settings -> Accelerator to a TPU VM, then rerun.')
    TORCH_XLA_VERSION = importlib.metadata.version('torch_xla')
    pins.append(f'torch_xla=={TORCH_XLA_VERSION}')
    os.environ.setdefault('PJRT_DEVICE', 'TPU')  # PyTorch/XLA runtime; inherited by the trainer.
    print('PyTorch:', torch.__version__, '| torch_xla:', TORCH_XLA_VERSION, '| torchvision:', TORCHVISION_VERSION)
else:
    TORCH_XLA_VERSION = None
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a GPU in Kaggle notebook settings, then rerun.')
    print('PyTorch:', torch.__version__, '| torchvision:', TORCHVISION_VERSION)
    for index in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(index)
        print(f'GPU {index}: {gpu.name}; {gpu.total_memory / 1024**3:.1f} GiB VRAM')

# Constraints prevent pip from replacing the working Kaggle PyTorch libraries.
constraints = TEMP_ROOT / 'torch-constraints.txt'
constraints.write_text('\n'.join(pins) + '\n')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--constraint', str(constraints),
    'monai==1.3.2', 'torchinfo==1.8.0', 'einops==0.8.0',
    'wandb', 'huggingface_hub', 'matplotlib', 'pandas', 'scikit-learn', 'tqdm',
], check=True)


PyTorch: 2.10.0+cu128 | torchvision: 0.25.0+cu128
GPU 0: Tesla T4; 14.6 GiB VRAM
GPU 1: Tesla T4; 14.6 GiB VRAM
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 1.9 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', '--constraint', '/kaggle/temp/fire-research/torch-constraints.txt', 'monai==1.3.2', 'torchinfo==1.8.0', 'einops==0.8.0', 'wandb', 'huggingface_hub', 'matplotlib', 'pandas', 'scikit-learn', 'tqdm'], returncode=0)

## 2. Clone the training code
Clones `REPO_BRANCH` of the GitHub repository into the notebook's code folder (replacing
any earlier clone), checks the trainer's CLI (`python -m firepred.cli.train --help`) and makes the
`firepred` package importable here. The exact commit is printed and saved
beside the checkpoints. Nothing is pushed to GitHub.

In [12]:
import shutil

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)  # Always start from a clean checkout.
subprocess.run(['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, str(CODE_DIR)],
               check=True)
if REPO_COMMIT:
    subprocess.run(['git', '-C', str(CODE_DIR), 'checkout', '--quiet', REPO_COMMIT], check=True)
CODE_COMMIT = subprocess.run(['git', '-C', str(CODE_DIR), 'rev-parse', 'HEAD'],
                             capture_output=True, text=True, check=True).stdout.strip()
print(f'Training code: {REPO_URL} @ {REPO_BRANCH} {CODE_COMMIT}')

help_result = subprocess.run(
    [sys.executable, '-m', 'firepred.cli.train', '--help'],
    cwd=CODE_DIR, text=True, capture_output=True,
)
if help_result.returncode:
    print(help_result.stdout)
    print(help_result.stderr)
    help_result.check_returncode()
for flag in ('--epochs', '--num-workers', '--data-root', '--wandb-user',
             '--tessera', '--upload-checkpoints', '--loss', '--ce-weight',
             '--lambda-dice', '--lambda-ce', '--lr-schedule', '--test-every', '--model-option',
             '--device'):
    assert flag in help_result.stdout, f'Training script is missing {flag}; check REPO_BRANCH'
print('Training imports and command-line options passed.')

# Notebook helpers (data staging, window repo names) come from the cloned package.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

Training code: https://github.com/SamuelWu318/fire-research-and-innovation.git @ main d10fd2f67088524acb20447bc75b76a1ff76e1ae
Training imports and command-line options passed.


## 3. Authenticate W&B
This creates a small `kaggle-permission-check` run in `gt-fri/AFBAPred`.
The training subprocess inherits the key through its environment.


In [6]:
from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret(WANDB_SECRET)
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_DIR'] = str(OUTPUT_DIR)
os.environ['WANDB_CACHE_DIR'] = str(TEMP_ROOT / 'wandb-cache')
os.environ['WANDB_CONFIG_DIR'] = str(TEMP_ROOT / 'wandb-config')
os.environ['WANDB_DATA_DIR'] = str(TEMP_ROOT / 'wandb-staging')
wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                name='kaggle-permission-check', mode='online') as run:
    run.log({'permission_check': 1})
    print('Permission-check run:', run.url)

# Delete the permission-check run so it doesn't clutter gt-fri/AFBAPred.
import time

check_path = f'{WANDB_ENTITY}/{WANDB_PROJECT}/{run.id}'
for attempt in range(5):
    try:
        check_run = wandb.Api().run(check_path)
        if check_run.name != 'kaggle-permission-check':
            raise RuntimeError(f'{check_path} is not the permission-check run; not deleting it')
        check_run.delete()
        print('Deleted permission-check run:', check_path)
        break
    except RuntimeError:
        raise
    except Exception as error:  # a just-finished run can take a few seconds to appear in the API
        if attempt == 4:
            print(f'Could not delete {check_path} ({error}); delete it in the W&B UI if needed.')
        else:
            time.sleep(3)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Permission-check run: https://wandb.ai/gt-fri/AFBAPred/runs/nczdgjph


permission_check,▁
permission_check,1


## 4. Prepare data
Download from the same Hugging Face dataset as the Colab version, or use the
attached-data path configured above. Both `train/val/test` and
`data/train`, `data/val`, `data/test` ZIP layouts are supported.
Extraction checks disk space and reuses a completed extraction within this session.
For large archives, attaching an already-extracted Kaggle Dataset avoids keeping
both a ZIP and its extracted contents on the runtime disk.


With `USE_TESSERA = True`, `tessera.zip` is also downloaded (or read from the attached
TESSERA path) and a symlink view is built: every kept window archive plus `tessera/`.
Fires listed in `excluded_npz.txt` (no prior-year TESSERA) are left out of the view;
the extracted data itself is never modified, so both modes can reuse one extraction.
With `USE_TESSERA = False`, no TESSERA file is downloaded or read.

In [7]:
from huggingface_hub import hf_hub_download

from firepred.config.data import window_repo_for
from firepred.hub.staging import build_tessera_view, extract_data, extract_zip, find_data_root

if HF_DATASET is None:
    HF_DATASET = window_repo_for(TS_LENGTH, TS_INTERVAL)
print('Window data repo:', HF_DATASET)

def get_hf_token():
    try:
        return secrets.get_secret(HF_SECRET)
    except Exception:
        print('HF_TOKEN unavailable; attempting public dataset access.')
        return False

if ATTACHED_DATA_ROOT:
    DATA_ROOT = find_data_root(ATTACHED_DATA_ROOT)
else:
    if ATTACHED_ZIP:
        archive_path = Path(ATTACHED_ZIP)
    else:
        archive_path = Path(hf_hub_download(
            repo_id=HF_DATASET, repo_type='dataset', filename='data.zip',
            token=get_hf_token(), local_dir=str(DOWNLOAD_DIR),
        ))
    DATA_ROOT = extract_data(archive_path, EXTRACT_DIR)

# TESSERA is downloaded and read only in TESSERA mode.
if USE_TESSERA:
    if ATTACHED_TESSERA_DIR:
        TESSERA_DIR = Path(ATTACHED_TESSERA_DIR)
    else:
        if ATTACHED_TESSERA_ZIP:
            tessera_archive = Path(ATTACHED_TESSERA_ZIP)
        else:
            tessera_archive = Path(hf_hub_download(
                repo_id=TESSERA_DATASET, repo_type='dataset', filename='tessera.zip',
                token=get_hf_token(), local_dir=str(DOWNLOAD_DIR),
            ))
        TESSERA_DIR = extract_zip(tessera_archive, TESSERA_EXTRACT_DIR) / 'tessera'
    print('TESSERA files:', sum(1 for _ in TESSERA_DIR.glob('*.npz')), 'in', TESSERA_DIR)
    TRAIN_ROOT = build_tessera_view(DATA_ROOT, TESSERA_DIR, TESSERA_VIEW_DIR)
else:
    TRAIN_ROOT = DATA_ROOT

print('Data root:', DATA_ROOT, '| training root:', TRAIN_ROOT)
for split in ('train', 'val', 'test'):
    print(split, 'NPZ archives:', sum(1 for _ in (TRAIN_ROOT / split).glob('*.npz')))


data.zip:   0%|          | 0.00/8.94G [00:00<?, ?B/s]

Uncompressed data: 8.3 GiB; free disk: 1061.5 GiB
Data root: /kaggle/temp/fire-research/data-ts10-it3 | training root: /kaggle/temp/fire-research/data-ts10-it3
train NPZ archives: 84
val NPZ archives: 11
test NPZ archives: 20


## 5. Check shapes and GPU memory
`python -m firepred.cli.check data` inspects one training batch and one validation sample. With the smoke test enabled,
it also performs one disposable forward/backward/Adam step using the same model settings.
This temporary model is discarded; the full training run starts fresh.
It uses the same `USE_TESSERA` setting and `MODEL_CHANNELS` as training.
If CUDA runs out of memory, reduce `BATCH_SIZE` and rerun from the configuration cell.
The check runs on the GPU or CPU only; with `ACCELERATOR = 'tpu'` skip it (or set `--device cpu`).


In [8]:
# Shape check and, with RUN_GPU_SMOKE_TEST, one disposable training step (python -m firepred.cli.check data).
# Uncomment to run before the full experiment.
# check = [sys.executable, '-u', '-m', 'firepred.cli.check', 'data',
#          '--data-root', str(TRAIN_ROOT), '--batch', str(BATCH_SIZE), '-ts', str(TS_LENGTH),
#          '-m', MODEL_NAME, '-nh', str(NUM_HEADS), '-ed', str(FEATURE_SIZE), '-lr', str(LEARNING_RATE),
#          '--loss', LOSS, '--ce-weight', str(CE_WEIGHT[0]), str(CE_WEIGHT[1]),
#          '--lambda-dice', str(LAMBDA_DICE), '--lambda-ce', str(LAMBDA_CE)]
# check += [arg for key, value in MODEL_OPTIONS.items() for arg in ('--model-option', f'{key}={value!r}')]
# if LOSS == 'dicefocal':
#     check += ['--lambda-dice', str(LAMBDA_DICE), '--lambda-focal', str(LAMBDA_FOCAL), '--focal-gamma', str(FOCAL_GAMMA)]
#     check += ['--focal-alpha', str(FOCAL_ALPHA)] if FOCAL_ALPHA is not None else []
# if USE_TESSERA and TESSERA_INPUTS:
#     check.append('--tessera')
# if RUN_GPU_SMOKE_TEST:
#     check.append('--train-step')
# subprocess.run(check, cwd=CODE_DIR, check=True)

### Check data time to load

In [16]:
assert (CODE_DIR / 'firepred').is_dir(), 'Run the clone cell (section 2) first'
subprocess.run([sys.executable, '-m', 'firepred.cli.check', 'timing',
                '--data-root', str(TRAIN_ROOT), '-ts', str(TS_LENGTH)],
               cwd=CODE_DIR, check=True)

0.55 s to load one window


CompletedProcess(args=['/usr/bin/python3', '-c', '\nimport sys, time\nfrom satimg_dataset_processor.data_generator_pred_torch import FireDataset\nroot, ts = sys.argv[1], int(sys.argv[2])\nds = FireDataset(root + "/train", ts_length=ts, n_channel=27,\n                 target_is_single_day=True, use_augmentations=True)\nstart = time.time()\nfor i in range(10):\n    ds[i]\nprint(f"{(time.time() - start) / 10:.2f} s to load one window")\n', '/kaggle/temp/fire-research/data-ts10-it3', '10'], returncode=0)

## 6. Train
The subprocess uses explicit code and data paths, so notebook directory changes do
not affect it. Logs stream below. Checkpoints remain in the W&B run's directory under
`/kaggle/working/fire-research-output/wandb` even if training later fails.
Each launch starts fresh; this trainer does not implement training resume.
With `ACCELERATOR = 'tpu'` the first epoch includes XLA compilation (and one more compile for a smaller
last batch); later epochs reuse the compiled graphs.
To find cloud checkpoints, open the training run's **Artifacts** tab. Upload failures
stop training with an error; the newly saved local checkpoint remains on disk.

W&B charts per epoch: `Training/Validation <loss> — Next-Day Fire Growth`; with
`LOSS = 'dicece'` also `Training/Validation Weighted Dice Loss` and `... Weighted
Cross-Entropy Loss`; `Training/Validation F1 — Next-Day Fire Growth` and `— Background`;
`Training/Validation IoU — Next-Day Fire Growth` (the paper's IoU, Equation 2; logged, never trained on);
`Validation Mean IoU` and `Validation Mean Dice (F1)` (these average in the background class, so they
are not the paper's numbers). **New, not in the paper:** with `TEST_EVERY > 0`, `Test F1/IoU — Next-Day
Fire Growth (periodic, current weights)` every `TEST_EVERY` epochs; logging only. Dice+CE totals are around 220 because
MONAI applies the 446.78 class weight to the Dice term too, so compare runs with different
losses by F1/IoU, not by loss value. After the last epoch the run reloads its best
checkpoint (lowest validation loss), scores the 24 test fires and logs `Test F1 — Next-Day Fire Growth`,
`Test IoU — Next-Day Fire Growth`, `Test F1 — Background`, `test_f1` and `test_iou` to the same run
(also in its summary). Pass `--no-test-after-train` to skip that step.

With TESSERA inputs, checkpoint names end in `_tessera.pth` and record the TESSERA
normalization; test such a checkpoint with `--tessera` and the same training root.
Dice+CE checkpoints end in `_dicece.pth`.

In [ ]:
import json

command = [
    sys.executable, '-u',                       # Unbuffered output in the notebook.
    '-m', 'firepred.cli.train',                # The trainer (firepred/training/run.py).
    '-m', MODEL_NAME,                          # Registered model (firepred/models/zoo/).
    '-mode', 'pred',                           # Prediction experiment label.
    '-b', str(BATCH_SIZE),                     # Windows per batch.
    '-r', '1',                                 # Required legacy run number.
    '-lr', str(LEARNING_RATE),                 # Adam learning rate.
    '-nh', str(NUM_HEADS),                     # Heads per Swin stage.
    '-ed', str(FEATURE_SIZE),                  # Base feature width.
    '-nc', str(MODEL_CHANNELS),                # 43, or 172 with TESSERA inputs.
    '-ts', str(TS_LENGTH),                     # Input days per window.
    '-it', str(TS_INTERVAL),                   # Window stride (recorded for the run).
    '-seed', '42',                             # Random seed.
    '--epochs', str(EPOCHS),                   # Full training passes.
    '--num-workers', str(NUM_WORKERS),         # CPU data-loading workers.
    '--device', DEVICE,                        # 'cuda' (GPU) or 'tpu' (PyTorch/XLA, one TPU device).
    '--unzip-dir', UNZIP_DIR,                  # 'off' reads archives; 'auto' unzips once.
    '--data-root', str(TRAIN_ROOT),            # train/val/test parent (TESSERA view in TESSERA mode).
    '--wandb-user', WANDB_ENTITY,              # W&B team: gt-fri.
    '--upload-checkpoints',                    # Best checkpoints -> W&B model Artifacts.
    '--loss', LOSS,                            # Loss; dicece / dicefocal also log their parts to W&B.
]
if LOSS == 'dicece':
    command += [
        '--ce-weight', str(CE_WEIGHT[0]), str(CE_WEIGHT[1]),  # Class weights (background, fire).
        '--lambda-dice', str(LAMBDA_DICE),     # Dice term weight.
        '--lambda-ce', str(LAMBDA_CE),         # Cross-entropy term weight.
    ]
if LOSS == 'dicefocal':
    command += ['--lambda-dice', str(LAMBDA_DICE), '--lambda-focal', str(LAMBDA_FOCAL),
                '--focal-gamma', str(FOCAL_GAMMA)]
    if FOCAL_ALPHA is not None:
        command += ['--focal-alpha', str(FOCAL_ALPHA)]
# Model-specific settings for the registry builder.
command += [arg for key, value in MODEL_OPTIONS.items() for arg in ('--model-option', f'{key}={value!r}')]
# NEW, not in the TS-SatFire paper or the official code: optional learning-rate decay.
command += [
    '--lr-schedule', LR_SCHEDULE,
    '--lr-decay-factor', str(LR_DECAY_FACTOR),
    '--lr-patience', str(LR_PATIENCE),
    '--lr-min', str(LR_MIN),
]
# NEW, not in the paper: test scores every TEST_EVERY epochs (logging only).
command += ['--test-every', str(TEST_EVERY)]
if USE_TESSERA and TESSERA_INPUTS:
    command.append('--tessera')                # Reads TRAIN_ROOT/tessera.
# Save reproducibility information without including secrets or environment values.
(OUTPUT_DIR / 'training-command.json').write_text(json.dumps(command, indent=2))
(OUTPUT_DIR / 'runtime.txt').write_text(
    f'Python: {sys.version}\nPyTorch: {torch.__version__}\n'
    f'torchvision: {TORCHVISION_VERSION}\ntorch_xla: {TORCH_XLA_VERSION}\n'
)
# Record the exact code version and data mode beside the checkpoints.
(OUTPUT_DIR / 'code-version.json').write_text(json.dumps({
    'repo': REPO_URL, 'branch': REPO_BRANCH, 'commit': CODE_COMMIT,
    'use_tessera': USE_TESSERA, 'tessera_inputs': USE_TESSERA and TESSERA_INPUTS,
    'model_channels': MODEL_CHANNELS, 'ts_length': TS_LENGTH, 'ts_interval': TS_INTERVAL,
    'model': MODEL_NAME, 'model_options': MODEL_OPTIONS, 'accelerator': ACCELERATOR,
    'loss': LOSS, 'learning_rate': LEARNING_RATE,
    'lr_schedule_new_not_in_paper': LR_SCHEDULE,
    'test_every_new_not_in_paper': TEST_EVERY,
    **({'lr_decay_factor': LR_DECAY_FACTOR, 'lr_patience': LR_PATIENCE} if LR_SCHEDULE == 'plateau' else {}),
    **({'lr_min': LR_MIN} if LR_SCHEDULE != 'none' else {}),
    **({'ce_weight': list(CE_WEIGHT), 'lambda_dice': LAMBDA_DICE, 'lambda_ce': LAMBDA_CE}
       if LOSS == 'dicece' else {}),
    **({'lambda_dice': LAMBDA_DICE, 'lambda_focal': LAMBDA_FOCAL, 'focal_gamma': FOCAL_GAMMA,
        'focal_alpha': FOCAL_ALPHA} if LOSS == 'dicefocal' else {}),
    'window_data': HF_DATASET,
}, indent=2))
subprocess.run(command, cwd=CODE_DIR, check=True)

print('Saved checkpoints:')
for checkpoint in sorted(OUTPUT_DIR.rglob('*.pth')):
    print(checkpoint)
print('Output directory:', OUTPUT_DIR)


## Notes and references
- `WANDB` and `HF_TOKEN` are secret **labels**, not values to paste into this notebook.
- Keep `use_checkpoint=False`: the custom Swin checkpoint branch needs a separate fix.
- Training tests its own best checkpoint at the end. A separate `-test` run is only needed for an
  older checkpoint: pass `--checkpoint` with its path, `--tessera` exactly when it was trained with
  TESSERA inputs, and `--swin-temporal-downsample` when it was trained before the paper's
  no-time-merging change (the trainer refuses a mismatch).
- The trainer's validation metric aggregation and lack of autocast are unchanged.
  A successful smoke test verifies one step, not an entire 100-epoch run.
- No Kaggle GPU or private-data execution was performed when this conversion was built.
- `ACCELERATOR = 'tpu'` was tested only against a stand-in for `torch_xla`, not on a Kaggle TPU yet.

References: [Kaggle Secrets implementation](https://github.com/Kaggle/docker-python/blob/main/patches/kaggle_secrets.py),
[Kaggle Python environment](https://github.com/Kaggle/docker-python),
[Hugging Face downloads](https://huggingface.co/docs/huggingface_hub/guides/download).
[W&B model Artifacts and upload completion](https://docs.wandb.ai/models/ref/python/experiments/artifact).
